# 02 Disposable Income Model

**Purpose:** Combine the cleaned tables into one model table and calculate weekly disposable income for international students across 5 cities.

**Inputs:** `data/clean/*.csv`
**Output:** `data/clean/model_base.csv`

**Formula:** Disposable income = net income − (rent + transport + OSHC + food + utilities and phone)

In [1]:
import itertools
import pandas as pd

clean = "../data/clean/"
names = ["fair_work_rates", "rent", "tax_rates", "lito", "insurance",
         "transport", "food_scenarios", "living_costs_other"]

# Only truly empty cells are treated as missing
tables = {n: pd.read_csv(clean + n + ".csv", keep_default_na=False, na_values=[""])
          for n in names}

for n, df in tables.items():
    print(f"{n}: {df.shape[0]} rows, {df.shape[1]} columns")

fair_work_rates: 22 rows, 5 columns
rent: 15 rows, 6 columns
tax_rates: 8 rows, 6 columns
lito: 4 rows, 6 columns
insurance: 5 rows, 6 columns
transport: 13 rows, 12 columns
food_scenarios: 16 rows, 5 columns
living_costs_other: 6 rows, 7 columns


## Assumptions
- 24 working hours per week (12 weekday, 6 Saturday, 6 Sunday), 52 weeks per year, Level 1 casual award rates
- Hospitality = Restaurant Industry Award, Retail = General Retail Industry Award
- Share room uses Flatmates averages; living alone uses SQM "All units" as a proxy
- Tax: resident (with low income tax offset) or foreign resident; Medicare levy excluded (assumes exemption with OSHC)
- All other costs at base scenario; OSHC is the mean of five providers

### Assumptions as variables

In [2]:
HOURS = {"Weekday": 12, "Saturday": 6, "Sunday": 6}   # 24 hours per week
LEVEL = "Level 1"
WEEKS = 52
AWARD = {"Hospitality": "Restaurant Industry Award",
         "Retail": "General Retail Industry Award"}
HOUSING = {"Share room": "Share room", "Living alone": "All units"}
RENT_SHOCKS = [0.0, 0.10]
CITIES = ["Sydney", "Melbourne", "Brisbane", "Perth", "Adelaide"]

### Tax functions and tests

In [3]:
def income_tax(income, residency):
    t = tables["tax_rates"]
    t = t[t["residency"] == residency].copy()
    t["threshold"] = (t["income_from"] - 1).clip(lower=0)
    row = t[t["threshold"] < income].iloc[-1]
    return row["base_tax"] + row["rate"] * (income - row["threshold"])

def lito(income):
    t = tables["lito"].copy()
    t["threshold"] = t["income_from"] - 1
    row = t[t["threshold"] < income].iloc[-1]
    return max(row["max_offset"] - row["reduction_rate"] * (income - row["reduction_starts_at"]), 0)

def net_income(income, residency):
    tax = income_tax(income, residency)
    if residency == "Resident":
        tax = max(tax - lito(income), 0)   # offset cannot create a refund
    return income - tax

# Tests against hand calculations
assert abs(net_income(43380, "Resident") - 40009) < 0.5
assert abs(net_income(43380, "Foreign resident") - 30366) < 0.5
assert net_income(10000, "Resident") == 10000   # below tax-free threshold
print("Tax function tests passed")

Tax function tests passed


### Inputs per city

In [4]:
fw = tables["fair_work_rates"]
rent = tables["rent"]

transport = (tables["transport"][tables["transport"]["scenario"] == "base"]
             .set_index("city")["weekly_cost_aud"])
food = (tables["food_scenarios"][tables["food_scenarios"]["scenario"] == "base"]
        .set_index("city")["food_weekly_aud"])
oshc_weekly = tables["insurance"]["weekly_aud"].mean()
other_weekly = tables["living_costs_other"].query("scenario == 'base'")["weekly_aud"].sum()

def hourly(industry, day_type):
    m = (fw["award"] == AWARD[industry]) & (fw["level"] == LEVEL) & (fw["day_type"] == day_type)
    return fw.loc[m, "casual_hourly_rate"].iloc[0]

def weekly_rent(city, housing):
    m = (rent["city"] == city) & (rent["housing_type"] == HOUSING[housing])
    return rent.loc[m, "weekly_rent"].iloc[0]

print("OSHC weekly:", round(oshc_weekly, 2), "| utilities + phone:", round(other_weekly, 2))
print(transport.to_dict())
print(food.to_dict())

OSHC weekly: 13.99 | utilities + phone: 49.61
{'Melbourne': 28.5, 'Sydney': 50.0, 'Brisbane': 5.0, 'Adelaide': 15.4, 'Perth': 14.4}
{'Sydney': 119.0, 'Melbourne': 119.0, 'Brisbane': 119.0, 'Perth': 119.0, 'Adelaide': 189.0}


### Build the model table

In [5]:
rows = []
for city, industry, housing, residency, shock in itertools.product(
        CITIES, AWARD, HOUSING, ["Resident", "Foreign resident"], RENT_SHOCKS):
    gross_week = sum(hourly(industry, d) * h for d, h in HOURS.items())
    net_week = net_income(gross_week * WEEKS, residency) / WEEKS
    rent_week = weekly_rent(city, housing) * (1 + shock)
    essentials = rent_week + transport[city] + oshc_weekly + food[city] + other_weekly
    rows.append({
        "city": city, "industry": industry, "housing": housing,
        "residency": residency, "rent_shock": shock,
        "gross_weekly": round(gross_week, 2),
        "net_weekly": round(net_week, 2),
        "rent_weekly": round(rent_week, 2),
        "transport_weekly": transport[city],
        "oshc_weekly": round(oshc_weekly, 2),
        "food_weekly": food[city],
        "utilities_phone_weekly": round(other_weekly, 2),
        "essentials_weekly": round(essentials, 2),
        "disposable_weekly": round(net_week - essentials, 2),
    })

model = pd.DataFrame(rows)
model["rent_to_net_ratio"] = (model["rent_weekly"] / model["net_weekly"]).round(3)
print(model.shape)   # expect (80, 15)
model.head()

(80, 15)


,city,industry,housing,residency,rent_shock,gross_weekly,net_weekly,rent_weekly,transport_weekly,oshc_weekly,food_weekly,utilities_phone_weekly,essentials_weekly,disposable_weekly,rent_to_net_ratio
0,Sydney,Hospitality,Share room,Resident,0.0,872.52,799.21,537.00,50.0,13.99,119.0,49.61,769.60,29.61,0.672
1,Sydney,Hospitality,Share room,Resident,0.1,872.52,799.21,590.70,50.0,13.99,119.0,49.61,823.30,-24.09,0.739
2,Sydney,Hospitality,Share room,Foreign resident,0.0,872.52,610.76,537.00,50.0,13.99,119.0,49.61,769.60,-158.84,0.879
3,Sydney,Hospitality,Share room,Foreign resident,0.1,872.52,610.76,590.70,50.0,13.99,119.0,49.61,823.30,-212.54,0.967
4,Sydney,Hospitality,Living alone,Resident,0.0,872.52,799.21,747.59,50.0,13.99,119.0,49.61,980.19,-180.98,0.935


### Validation

In [6]:
diff = (model["disposable_weekly"] - (model["net_weekly"] - model["essentials_weekly"])).abs()
print("Max difference:", round(diff.max(), 4))

Max difference: 0.01


In [7]:
import numpy as np

# Structural checks
assert model.shape == (80, 15)
assert model.notna().all().all()
assert np.isclose(model["disposable_weekly"],
                  model["net_weekly"] - model["essentials_weekly"],
                  atol=0.02).all()

# Compare against the hand-checked results (Hospitality, Resident, no rent shock)
check = model.query("industry == 'Hospitality' and residency == 'Resident' and rent_shock == 0")
check.pivot(index="city", columns="housing", values="disposable_weekly")

housing,Living alone,Share room
city,,
Adelaide,-14.98,200.21
Brisbane,-40.76,169.61
Melbourne,-2.32,195.11
Perth,-65.43,168.21
Sydney,-180.98,29.61


In [8]:
model.to_csv(clean + "model_base.csv", index=False)
print("Saved model_base.csv")

Saved model_base.csv
